In [ ]:
# 1. 一鍵安裝必要套件
!pip install -q gradio plotly pandas numpy

import gradio as gr
import plotly.graph_objects as go
import numpy as np
import traceback

# ==================== 【小工具功能：必須放在最前面防錯】 ====================

def format_seconds_to_pace(seconds):
    """將純秒數轉換為高質感的分:秒格式字串"""
    mins = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{mins}:{secs:02d}"

def update_run_type_visibility(sport):
    """當運動項目選跑步時，才顯示跑步類型選單"""
    return gr.update(visible=("跑步" in sport))

# ==================== 【後端核心：科學化運動算法大腦】 ====================

def calculate_pace_and_structure(sport, run_type, fatigue, total_time):
    try:
        # 基礎有氧配速秒數基準（疲勞度越高，配速越慢）
        base_seconds = 300 + (fatigue * 15)

        chart_y_label = "配速 (每公里幾分鐘)"
        unit = "分/公里"

        # 核心項目分流：跑步、騎車、游泳
        if "跑步" in sport:
            chart_y_label = "跑步配速 (分/公里)"
            unit = "分/公里"

            # 安全機制：高度疲勞強制轉恢復
            if fatigue >= 8:
                ai_analysis = f"🚨 【體能導師警告】\n偵測到目前疲勞指標極高 ({fatigue}/10)。今日不宜進行任何高強度衝刺，系統已自動介入，課表變更為「動態恢復慢跑」，專注於放鬆緊繃肌群。"
                run_type = "有氧基礎跑 (E 跑)"

            if run_type == "有氧基礎跑 (E 跑)":
                warm_t = int(total_time * 0.15)
                main_t = int(total_time * 0.7)
                cool_t = total_time - warm_t - main_t
                p_warm, p_main, p_cool = base_seconds + 30, base_seconds, base_seconds + 45

                if fatigue < 8:
                    ai_analysis = f"✅ 【體能狀態評估】\n當前體能狀態穩定 ({fatigue}/10)。今日執行基礎有氧 E 跑，此課表目的在於穩定建立心肺微血管密度。請保持在可以流暢聊天的體感強度。"

                warm_detail = f"⏱️ {warm_t} 分鐘\n● 配速：{format_seconds_to_pace(p_warm)} {unit}\n- 輕鬆慢跑開始，漸進喚醒下肢關節。"
                main_detail = f"⏱️ {main_t} 分鐘\n● 配速：{format_seconds_to_pace(p_main)} {unit}\n- 核心有氧巡航，維持穩定呼吸節奏。"
                cooldown_detail = f"⏱️ {cool_t} 分鐘\n● 配速：{format_seconds_to_pace(p_cool)} {unit}\n- 極慢跑協議，搭配下肢動態放鬆。"

                x_timeline = ['準備熱身', '熱身結束', '主段開始', '主段結束', '緩和開始', '訓練結束']
                y_timeline = [p_warm/60, p_warm/60, p_main/60, p_main/60, p_cool/60, p_cool/60]

            elif run_type == "馬拉松配速跑 (M 跑)":
                warm_t = 15
                main_t = int(total_time * 0.7)
                cool_t = total_time - warm_t - main_t
                p_warm, p_main, p_cool = base_seconds + 20, base_seconds - 20, base_seconds + 40

                ai_analysis = f"🏃‍♂️ 【體能狀態評估】\n今日目標為 M 跑模擬協議。主要鍛鍊身體在馬拉松目標速度下的能量代謝效率，對於心智耐力的培養至關重要。"

                warm_detail = f"⏱️ {warm_t} 分鐘\n● 配速：{format_seconds_to_pace(p_warm)} {unit}\n- 慢跑熱身，讓身體微微出汗。"
                main_detail = f"⏱️ {main_t} 分鐘\n● 配速：{format_seconds_to_pace(p_main)} {unit}\n- 連續定速主訓練，模擬賽事節奏。"
                cooldown_detail = f"⏱️ {cool_t} 分鐘\n● 配速：{format_seconds_to_pace(p_cool)} {unit}\n- 慢走調節心率，排除肌肉緊繃。"

                x_timeline = ['準備熱身', '熱身結束', '主段開始', '主段結束', '緩和開始', '訓練結束']
                y_timeline = [p_warm/60, p_warm/60, p_main/60, p_main/60, p_cool/60, p_cool/60]

            elif run_type == "乳酸閥值/節奏跑 (T 跑)":
                warm_t = 15
                main_t = int(total_time * 0.6)
                cool_t = total_time - warm_t - main_t
                p_warm, p_main, p_cool = base_seconds + 15, base_seconds - 40, base_seconds + 45

                ai_analysis = f"🔥 【體能狀態評估】\n今日進入 T 跑協議（舒服的痛苦）。目的是提高乳酸閥值臨界點，確保高速續航力的鍛鍊。"

                warm_detail = f"⏱️ {warm_t} 分鐘\n● 配速：{format_seconds_to_pace(p_warm)} {unit}\n- 包含 10分鐘慢跑 + 3趟漸速跑。"
                main_detail = f"⏱️ {main_t} 分鐘\n● 結構：維持 {format_seconds_to_pace(p_main)} {unit}\n- 連續維持 20-30 分鐘，或執行巡航間歇。"
                cooldown_detail = f"⏱️ {cool_t} 分鐘\n● 配速：{format_seconds_to_pace(p_cool)} {unit}\n- 慢跑排解高乳酸環境。"

                x_timeline = ['準備熱身', '熱身結束', 'T跑主段開始', 'T跑主段結束', '緩和開始', '訓練結束']
                y_timeline = [p_warm/60, p_warm/60, p_main/60, p_main/60, p_cool/60, p_cool/60]

            else:  # 最大攝氧量間歇跑 (I 跑) —— 🛠️ 完美鋸齒降速波形
                warm_t = 15
                main_t = int(total_time * 0.5)
                cool_t = total_time - warm_t - main_t

                p_warm = base_seconds + 15
                p_fast = base_seconds - 75
                p_rest = base_seconds + 90
                p_cool = base_seconds + 60

                single_lap_min = p_fast / 60
                lap_count = max(2, int(main_t / (single_lap_min * 2)))

                ai_analysis = f"⚡ 【體能狀態評估】\n進入高強度 I 跑間歇協議！【核心標準】工作與休息時間比為 1:1，衝刺後速度必須大幅降下來進行積極恢復，確保下一趟的爆發力。"

                warm_detail = f"⏱️ {warm_t} 分鐘\n● 配速：{format_seconds_to_pace(p_warm)} {unit}\n- 完整熱身，完全喚醒心肺。"
                main_detail = f"⏱️ {main_t} 分鐘\n● 結構：亞索重複組 —— 共 {lap_count} 趟\n- 🚀 衝刺：{format_seconds_to_pace(p_fast)} {unit}\n- 🛑 恢復：{format_seconds_to_pace(p_rest)} {unit}\n- 嚴格執行 1:1 降速恢復！"
                cooldown_detail = f"⏱️ {cool_t} 分鐘\n● 配速：{format_seconds_to_pace(p_cool)} {unit}\n- 極慢走協助排除深層乳酸。"

                x_timeline = ['準備熱身', '熱身結束']
                y_timeline = [p_warm/60, p_warm/60]
                for i in range(1, lap_count + 1):
                    x_timeline.extend([f'第{i}趟衝刺', f'第{i}趟恢復'])
                    y_timeline.extend([p_fast/60, p_rest/60])
                x_timeline.extend(['緩和開始', '訓練結束'])
                y_timeline.extend([p_cool/60, p_cool/60])

        elif "騎車" in sport:
            chart_y_label = "單車目標功率 (Watts)"
            unit = "瓦數 (W)"
            base_power = 220 - (fatigue * 12)

            if fatigue >= 8:
                ai_analysis = f"🚨 【體能導師警告】\n單車學員疲勞度高 ({fatigue}/10)。今日強制降階為「有氧輕踩協議」，控制在 Zone 1 範圍。"
                p_warm, p_main, p_cool = int(base_power * 0.6), int(base_power * 0.7), int(base_power * 0.5)
            else:
                ai_analysis = f"🚴‍♂️ 【體能狀態評估】\n單車儲備狀態優良。主段將拉高至 {base_power}W，請專注於高迴轉速的圓滑踩踏。"
                p_warm, p_main, p_cool = int(base_power * 0.6), base_power, int(base_power * 0.5)

            warm_detail = f"⏱️ {int(total_time*0.2)} 分鐘\n● 強度：{p_warm}W\n- 輕齒比高迴轉熱身。"
            main_detail = f"⏱️ {int(total_time*0.65)} 分鐘\n● 強度：{p_main}W\n- 穩定功率巡航。"
            cooldown_detail = f"⏱️ {total_time - int(total_time*0.2) - int(total_time*0.65)} 分鐘\n● 強度：{p_cool}W\n- 降至最小阻力放鬆。"

            x_timeline = ['熱身開始', '熱身結束', '主段功率Cruise', '主段結束', '阻力放鬆', '騎乘結束']
            y_timeline = [p_warm, p_warm, p_main, p_main, p_cool, p_cool]

        else:  # 游泳
            chart_y_label = "游泳配速 (分:秒/100m)"
            unit = "分:秒/100m"
            base_swim_seconds = 100 + (fatigue * 5)
            p_warm, p_main, p_cool = base_swim_seconds + 15, base_swim_seconds, base_swim_seconds + 20

            if fatigue >= 8:
                ai_analysis = f"🚨 【體能導師警告】\n游泳學員高疲勞警示。主訓練課表變更為「放鬆游協議」，尋找低阻力的精準滑行體感。"
            else:
                ai_analysis = f"🏊‍♂️ 【體能狀態評估】\n游泳體能狀態良好。主段設定為有氧節奏（{format_seconds_to_pace(p_main)} /100m），專注於雙臂划水的高效延伸。"

            warm_detail = f"⏱️ {int(total_time*0.2)} 分鐘\n● 配速：{format_seconds_to_pace(p_warm)} /100m\n- 輕鬆任意式熱身。"
            main_detail = f"⏱️ {int(total_time*0.65)} 分鐘\n● 配速：{format_seconds_to_pace(p_main)} /100m\n- 穩定長距離包夾組。"
            cooldown_detail = f"⏱️ {total_time - int(total_time*0.2) - int(total_time*0.65)} 分鐘\n● 配速：{format_seconds_to_pace(p_cool)} /100m\n- 輕鬆仰式或蛙式緩和。"

            x_timeline = ['入水熱身', '熱身結束', '有氧主段開始', '主段結束', '游降速組', '訓練結束']
            y_timeline = [p_warm/60, p_warm/60, p_main/60, p_main/60, p_cool/60, p_cool/60]

        # 繪製 Plotly 專業圖表
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x_timeline,
            y=y_timeline,
            mode='lines+markers',
            line=dict(shape='linear', width=3, color='#D4AF37'),
            marker=dict(size=8, color='#F3C65F', line=dict(width=1.5, color='#1A1919')),
            fill='tozeroy',
            fillcolor='rgba(212, 175, 55, 0.08)',
            hovertemplate="<b>%{x}</b><br>目標強度: %{y:.2f} " + unit.split(' ')[0] + "<extra></extra>"
        ))

        fig.update_layout(
            title=dict(text=f"<b>{sport} - 訓練強度動態分佈圖</b>", font=dict(size=14, color='#E2E8F0')),
            template='plotly_dark',
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)',
            font=dict(color='#A0AEC0', family='sans-serif'),
            margin=dict(l=55, r=40, t=50, b=40),
            xaxis=dict(showgrid=False, zeroline=False, tickfont=dict(size=10, color='#E2E8F0')),
            yaxis=dict(title=chart_y_label, showgrid=True, gridcolor='rgba(255,255,255,0.05)', gridwidth=1, zeroline=False, tickfont=dict(size=10, color='#E2E8F0')),
            hovermode='x unified',
            hoverlabel=dict(bgcolor="rgba(26, 25, 25, 0.95)", font_size=13, bordercolor="rgba(212, 175, 55, 0.4)")
        )

        # 🛠️ 【正確隱藏工具列做法】：改用語法加進後端 Layout
        fig.update_layout(modebar_remove=['zoom', 'pan', 'select', 'lasso', 'zoomIn', 'zoomOut', 'autoScale', 'resetScale'])

        if "跑步" in sport or "游泳" in sport:
            fig.update_yaxes(autorange="reversed")

        return ai_analysis, warm_detail, main_detail, cooldown_detail, fig

    except Exception as e:
        err_fig = go.Figure()
        err_fig.update_layout(template='plotly_dark', title="圖表生成失敗", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
        return f"系統發生錯誤：\n{str(e)}", "計算失敗", "計算失敗", "計算失敗", err_fig

# ==================== 【Gradio UI 介面設計與自訂 CSS】 ====================

liquid_glass_css = """
body, .gradio-container {
    background-color: #1A1919 !important;
    background-image: radial-gradient(circle at 50% 0%, #242120 0%, #1A1919 80%) !important;
    color: #E2E8F0 !important;
    font-family: system-ui, -apple-system, sans-serif !important;
}
.gr-box, .gr-panel, .gr-form { border: none !important; background: transparent !important; box-shadow: none !important; }

/* 強制隱藏所有原生工具列與右上角叉叉 */
button[aria-label="Clear"], button[aria-label="Copy"], .gr-button-icon, .close-button {
    display: none !important;
}

.glass-panel {
    background: rgba(255, 255, 255, 0.015) !important;
    backdrop-filter: blur(16px) !important;
    border: 1px solid rgba(255, 255, 255, 0.03) !important;
    border-top: 1px solid rgba(255, 255, 255, 0.07) !important;
    border-radius: 16px !important;
    padding: 20px !important;
    box-shadow: 0 12px 40px rgba(0, 0, 0, 0.4) !important;
}

.glass-card {
    background: linear-gradient(135deg, rgba(255, 255, 255, 0.025) 0%, rgba(255, 255, 255, 0.005) 100%) !important;
    backdrop-filter: blur(12px) !important;
    border: 1px solid rgba(255, 255, 255, 0.03) !important;
    border-top: 1px solid rgba(212, 175, 55, 0.18) !important;
    border-radius: 10px !important;
    padding: 14px !important;
}

.ai-box {
    background: rgba(212, 175, 55, 0.015) !important;
    border: 1px solid rgba(212, 175, 55, 0.1) !important;
    border-radius: 10px !important;
    font-size: 14px !important;
    color: #F6F0E5 !important;
    padding: 12px !important;
}

.accent-btn {
    background: linear-gradient(135deg, #D4AF37 0%, #AA6C39 100%) !important;
    color: #1A1919 !important;
    border: none !important;
    border-radius: 6px !important;
    font-weight: 600 !important;
}
.accent-btn:hover {
    background: linear-gradient(135deg, #E5C158 0%, #BD7D4A 100%) !important;
    transform: translateY(-1px) !important;
}

input[type="range"]::-webkit-slider-thumb { background: #D4AF37 !important; }
"""

# 🛠️ 【穩定度金牌寫法】：theme 與 css 共同傳入 Blocks 機制，兼顧安全防錯與舊版相容性
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="amber", neutral_hue="stone"),
    css=liquid_glass_css
) as demo:

    gr.Markdown("## 訓練課表產生器", elem_classes="gr-box")

    with gr.Row():
        # 【左側：參數面板】
        with gr.Column(scale=1, elem_classes="glass-panel"):
            gr.Markdown("#### 參數設定")
            inp_sport = gr.Radio(["跑步 (Running)", "騎車 (Cycling)", "游泳 (Swimming)"], label="運動項目", value="跑步 (Running)")
            inp_run_type = gr.Dropdown(["有氧基礎跑 (E 跑)", "馬拉松配速跑 (M 跑)", "乳酸閥值/節奏跑 (T 跑)", "最大攝氧量間歇跑 (I 跑)"], label="跑步訓練類型", value="有氧基礎跑 (E 跑)", visible=True)
            inp_fatigue = gr.Slider(minimum=1, maximum=10, step=1, label="今日疲勞度指標 (1-10)", value=3)
            inp_time = gr.Slider(minimum=20, maximum=180, step=5, label="預計總時間 (分鐘)", value=60)
            btn_submit = gr.Button("生成訓練計畫", elem_classes="accent-btn")

        inp_sport.change(fn=update_run_type_visibility, inputs=inp_sport, outputs=inp_run_type)

        # 【右側：結果顯示面板】
        with gr.Column(scale=2, elem_classes="glass-panel"):
            gr.Markdown("#### 系統體能診斷")
            out_ai = gr.Textbox(label="", lines=2, elem_classes="ai-box", interactive=False, show_copy_button=False)

            gr.Markdown("#### 結構化課表")
            with gr.Row():
                with gr.Column(scale=1, min_width=0):
                    out_warmup = gr.Textbox(label="1. 暖身階段", lines=6, elem_classes="glass-card", interactive=False, show_copy_button=False)
                with gr.Column(scale=1, min_width=0):
                    out_main = gr.Textbox(label="2. 主訓練階段", lines=6, elem_classes="glass-card", interactive=False, show_copy_button=False)
                with gr.Column(scale=1, min_width=0):
                    out_cooldown = gr.Textbox(label="3. 緩和階段", lines=6, elem_classes="glass-card", interactive=False, show_copy_button=False)

            gr.Markdown("#### 強度分佈趨勢")
            # 🛠️ 【核心修正點】：拿掉會噴錯的 config 參數，維持原生清爽渲染
            out_chart = gr.Plot(label="", show_label=False)

    # 1. 點擊按鈕事件
    btn_submit.click(
        fn=calculate_pace_and_structure,
        inputs=[inp_sport, inp_run_type, inp_fatigue, inp_time],
        outputs=[out_ai, out_warmup, out_main, out_cooldown, out_chart]
    )

    # 2. 頁面載入自動觸發
    demo.load(
        fn=calculate_pace_and_structure,
        inputs=[inp_sport, inp_run_type, inp_fatigue, inp_time],
        outputs=[out_ai, out_warmup, out_main, out_cooldown, out_chart]
    )

# 🛠️ 【核心修正】：`.launch()` 內保持無 UI 參數干擾狀態
demo.launch(share=True)

/tmp/ipykernel_7623/2875637112.py:250: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipykernel_7623/2875637112.py:250: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dc8e54b453bcfaa112.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# 1. 一鍵安裝必要套件
!pip install -q gradio plotly pandas numpy

import gradio as gr
import plotly.graph_objects as go
import numpy as np
import traceback

# ==================== 【小工具功能：必須放在最前面防錯】 ====================

def format_seconds_to_pace(seconds):
    """將純秒數轉換為高質感的分:秒格式字串"""
    mins = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{mins}:{secs:02d}"

def parse_pace_to_seconds(pace_str):
    """將使用者輸入的 '分:秒' 轉換為純秒數，若格式錯誤則給予預設值 330 秒 (05:30)"""
    try:
        if ":" in pace_str:
            parts = pace_str.split(":")
            return int(parts[0]) * 60 + int(parts[1])
        else:
            # 預防只輸入數字的情況（例如輸入 5 改為 5分0秒）
            return int(pace_str) * 60
    except:
        return 330  # 發生任何例外時的防錯機制（預設 05:30）

def update_run_type_visibility(sport):
    """當運動項目選跑步時，才顯示跑步類型選單與個人配速輸入框"""
    is_run = "跑步" in sport
    return gr.update(visible=is_run), gr.update(visible=is_run)

# ==================== 【後端核心：科學化運動算法大腦】 ====================

def calculate_pace_and_structure(sport, run_type, fatigue, total_time, personal_pace_str):
    try:
        # 解析個人馬拉松配速基準（例如輸入 05:30 -> 330 秒）
        user_base_seconds = parse_pace_to_seconds(personal_pace_str)

        # 基礎有氧配速秒數基準（引入個人配速：疲勞度越高，配速越慢）
        base_seconds = user_base_seconds + (fatigue * 10)

        chart_y_label = "配速 (每公里幾分鐘)"
        unit = "分/公里"

        # 核心項目分流：跑步、騎車、游泳
        if "跑步" in sport:
            chart_y_label = "跑步配速 (分/公里)"
            unit = "分/公里"

            # 安全機制：高度疲勞強制轉恢復
            if fatigue >= 8:
                ai_analysis = f"🚨 【體能導師警告】\n偵測到目前疲勞指標極高 ({fatigue}/10)。今日不宜進行任何高強度衝刺，系統已自動介入，課表變更為「動態恢復慢跑」，專注於放鬆緊繃肌群。"
                run_type = "有氧基礎跑 (E 跑)"

            if run_type == "有氧基礎跑 (E 跑)":
                warm_t = int(total_time * 0.15)
                main_t = int(total_time * 0.7)
                cool_t = total_time - warm_t - main_t
                # E跑配速科學設定：比個人馬拉松配速慢 30~50 秒
                p_warm, p_main, p_cool = base_seconds + 50, base_seconds + 30, base_seconds + 60

                if fatigue < 8:
                    ai_analysis = f"✅ 【體能狀態評估】\n依據您個人設定基底（{personal_pace_str}），今日執行基礎有氧 E 跑。此課表目的在於穩定建立心肺微血管密度。請保持在可以流暢聊天的體感強度。"

                warm_detail = f"⏱️ {warm_t} 分鐘\n● 配速：{format_seconds_to_pace(p_warm)} {unit}\n- 輕鬆慢跑開始，漸進喚醒下肢關節。"
                main_detail = f"⏱️ {main_t} 分鐘\n● 配速：{format_seconds_to_pace(p_main)} {unit}\n- 核心有氧巡航，維持穩定呼吸節奏。"
                cooldown_detail = f"⏱️ {cool_t} 分鐘\n● 配速：{format_seconds_to_pace(p_cool)} {unit}\n- 極慢跑協議，搭配下肢動態放鬆。"

                x_timeline = ['準備熱身', '熱身結束', '主段開始', '主段結束', '緩和開始', '訓練結束']
                y_timeline = [p_warm/60, p_warm/60, p_main/60, p_main/60, p_cool/60, p_cool/60]

            elif run_type == "馬拉松配速跑 (M 跑)":
                warm_t = 15
                main_t = int(total_time * 0.7)
                cool_t = total_time - warm_t - main_t
                # M跑配速科學設定：精準對齊個人馬拉松目標配速
                p_warm, p_main, p_cool = base_seconds + 30, base_seconds, base_seconds + 45

                ai_analysis = f"🏃‍♂️ 【體能狀態評估】\n今日目標為 M 跑模擬協議。主要鍛鍊身體在您目標速度（{personal_pace_str}）下的能量代謝效率，對於心智耐力的培養至關重要。"

                warm_detail = f"⏱️ {warm_t} 分鐘\n● 配速：{format_seconds_to_pace(p_warm)} {unit}\n- 慢跑熱身，讓身體微微出汗。"
                main_detail = f"⏱️ {main_t} 分鐘\n● 配速：{format_seconds_to_pace(p_main)} {unit}\n- 連續定速主訓練，模擬賽事節奏。"
                cooldown_detail = f"⏱️ {cool_t} 分鐘\n● 配速：{format_seconds_to_pace(p_cool)} {unit}\n- 慢走調節心率，排除肌肉緊繃。"

                x_timeline = ['準備熱身', '熱身結束', '主段開始', '主段結束', '緩和開始', '訓練結束']
                y_timeline = [p_warm/60, p_warm/60, p_main/60, p_main/60, p_cool/60, p_cool/60]

            elif run_type == "乳酸閥值/節奏跑 (T 跑)":
                warm_t = 15
                main_t = int(total_time * 0.6)
                cool_t = total_time - warm_t - main_t
                # T跑配速科學設定：比個人馬拉松配速快 20~30 秒
                p_warm, p_main, p_cool = base_seconds + 15, base_seconds - 25, base_seconds + 45

                ai_analysis = f"🔥 【體能狀態評估】\n今日進入 T 跑協議（舒服的痛苦）。目標配速已鎖定比平常更快的 {format_seconds_to_pace(p_main)}。目的是提高乳酸閥值臨界點，確保高速續航力的鍛鍊。"

                warm_detail = f"⏱️ {warm_t} 分鐘\n● 配速：{format_seconds_to_pace(p_warm)} {unit}\n- 包含 10分鐘慢跑 + 3趟漸速跑。"
                main_detail = f"⏱️ {main_t} 分鐘\n● 結構：維持 {format_seconds_to_pace(p_main)} {unit}\n- 連續維持 20-30 分鐘，或執行巡航間歇。"
                cooldown_detail = f"⏱️ {cool_t} 分鐘\n● 配速：{format_seconds_to_pace(p_cool)} {unit}\n- 慢跑排解高乳酸環境。"

                x_timeline = ['準備熱身', '熱身結束', 'T跑主段開始', 'T跑主段結束', '緩和開始', '訓練結束']
                y_timeline = [p_warm/60, p_warm/60, p_main/60, p_main/60, p_cool/60, p_cool/60]

            else:  # 最大攝氧量間歇跑 (I 跑) —— 🛠️ 完美鋸齒降速波形
                warm_t = 15
                main_t = int(total_time * 0.5)
                cool_t = total_time - warm_t - main_t

                # I跑配速科學設定：衝刺段比個人馬拉松配速大幅快 50~60 秒
                p_warm = base_seconds + 15
                p_fast = base_seconds - 55
                p_rest = base_seconds + 80
                p_cool = base_seconds + 60

                single_lap_min = p_fast / 60
                lap_count = max(2, int(main_t / (single_lap_min * 2)))

                ai_analysis = f"⚡ 【體能狀態評估】\n進入高強度 I 跑間歇協議！衝刺配速壓在極限的 {format_seconds_to_pace(p_fast)}。【核心標準】工作與休息時間比為 1:1，衝刺後速度必須大幅降下來進行積極恢復，確保下一趟的爆發力。"

                warm_detail = f"⏱️ {warm_t} 分鐘\n● 配速：{format_seconds_to_pace(p_warm)} {unit}\n- 完整熱身，完全喚醒心肺。"
                main_detail = f"⏱️ {main_t} 分鐘\n● 結構：亞索重複組 —— 共 {lap_count} 趟\n- 🚀 衝刺：{format_seconds_to_pace(p_fast)} {unit}\n- 🛑 恢復：{format_seconds_to_pace(p_rest)} {unit}\n- 嚴格執行 1:1 降速恢復！"
                cooldown_detail = f"⏱️ {cool_t} 分鐘\n● 配速：{format_seconds_to_pace(p_cool)} {unit}\n- 極慢走協助排除深層乳酸。"

                x_timeline = ['準備熱身', '熱身結束']
                y_timeline = [p_warm/60, p_warm/60]
                for i in range(1, lap_count + 1):
                    x_timeline.extend([f'第{i}趟衝刺', f'第{i}趟恢復'])
                    y_timeline.extend([p_fast/60, p_rest/60])
                x_timeline.extend(['緩和開始', '訓練結束'])
                y_timeline.extend([p_cool/60, p_cool/60])

        elif "騎車" in sport:
            chart_y_label = "單車目標功率 (Watts)"
            unit = "瓦數 (W)"
            base_power = 220 - (fatigue * 12)

            if fatigue >= 8:
                ai_analysis = f"🚨 【體能導師警告】\n單車學員疲勞度高 ({fatigue}/10)。今日強制降階為「有氧輕踩協議」，控制在 Zone 1 範圍。"
                p_warm, p_main, p_cool = int(base_power * 0.6), int(base_power * 0.7), int(base_power * 0.5)
            else:
                ai_analysis = f"🚴‍♂️ 【體能狀態評估】\n單車儲備狀態優良。主段將拉高至 {base_power}W，請專注於高迴轉速的圓滑踩踏。"
                p_warm, p_main, p_cool = int(base_power * 0.6), base_power, int(base_power * 0.5)

            warm_detail = f"⏱️ {int(total_time*0.2)} 分鐘\n● 強度：{p_warm}W\n- 輕齒比高迴轉熱身。"
            main_detail = f"⏱️ {int(total_time*0.65)} 分鐘\n● 強度：{p_main}W\n- 穩定功率巡航。"
            cooldown_detail = f"⏱️ {total_time - int(total_time*0.2) - int(total_time*0.65)} 分鐘\n● 強度：{p_cool}W\n- 降至最小阻力放鬆。"

            x_timeline = ['熱身開始', '熱身結束', '主段功率Cruise', '主段結束', '阻力放鬆', '騎乘結束']
            y_timeline = [p_warm, p_warm, p_main, p_main, p_cool, p_cool]

        else:  # 游泳
            chart_y_label = "游泳配速 (分:秒/100m)"
            unit = "分:秒/100m"
            base_swim_seconds = 100 + (fatigue * 5)
            p_warm, p_main, p_cool = base_swim_seconds + 15, base_swim_seconds, base_swim_seconds + 20

            if fatigue >= 8:
                ai_analysis = f"🚨 【體能導師警告】\n游泳學員高疲勞警示。主訓練課表變更為「放鬆游協議」，尋找低阻力的精準滑行體感。"
            else:
                ai_analysis = f"🏊‍♂️ 【體能狀態評估】\n游泳體能狀態良好。主段設定為有氧節奏（{format_seconds_to_pace(p_main)} /100m），專注於雙臂划水的高效延伸。"

            warm_detail = f"⏱️ {int(total_time*0.2)} 分鐘\n● 配速：{format_seconds_to_pace(p_warm)} /100m\n- 輕鬆任意式熱身。"
            main_detail = f"⏱️ {int(total_time*0.65)} 分鐘\n● 配速：{format_seconds_to_pace(p_main)} /100m\n- 穩定長距離包夾組。"
            cooldown_detail = f"⏱️ {total_time - int(total_time*0.2) - int(total_time*0.65)} 分鐘\n● 配速：{format_seconds_to_pace(p_cool)} /100m\n- 輕鬆仰式或蛙式緩和。"

            x_timeline = ['入水熱身', '熱身結束', '有氧主段開始', '主段結束', '游降速組', '訓練結束']
            y_timeline = [p_warm/60, p_warm/60, p_main/60, p_main/60, p_cool/60, p_cool/60]

        # 繪製 Plotly 專業圖表
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x_timeline,
            y=y_timeline,
            mode='lines+markers',
            line=dict(shape='linear', width=3, color='#D4AF37'),
            marker=dict(size=8, color='#F3C65F', line=dict(width=1.5, color='#1A1919')),
            fill='tozeroy',
            fillcolor='rgba(212, 175, 55, 0.08)',
            hovertemplate="<b>%{x}</b><br>目標強度: %{y:.2f} " + unit.split(' ')[0] + "<extra></extra>"
        ))

        fig.update_layout(
            title=dict(text=f"<b>{sport} - 訓練強度動態分佈圖</b>", font=dict(size=14, color='#E2E8F0')),
            template='plotly_dark',
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)',
            font=dict(color='#A0AEC0', family='sans-serif'),
            margin=dict(l=55, r=40, t=50, b=40),
            xaxis=dict(showgrid=False, zeroline=False, tickfont=dict(size=10, color='#E2E8F0')),
            yaxis=dict(title=chart_y_label, showgrid=True, gridcolor='rgba(255,255,255,0.05)', gridwidth=1, zeroline=False, tickfont=dict(size=10, color='#E2E8F0')),
            hovermode='x unified',
            hoverlabel=dict(bgcolor="rgba(26, 25, 25, 0.95)", font_size=13, bordercolor="rgba(212, 175, 55, 0.4)")
        )

        # 正確隱藏工具列做法
        fig.update_layout(modebar_remove=['zoom', 'pan', 'select', 'lasso', 'zoomIn', 'zoomOut', 'autoScale', 'resetScale'])

        if "跑步" in sport or "游泳" in sport:
            fig.update_yaxes(autorange="reversed")

        return ai_analysis, warm_detail, main_detail, cooldown_detail, fig

    except Exception as e:
        err_fig = go.Figure()
        err_fig.update_layout(template='plotly_dark', title="圖表生成失敗", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
        return f"系統發生錯誤：\n{str(e)}", "計算失敗", "計算失敗", "計算失敗", err_fig

# ==================== 【Gradio UI 介面設計與自訂 CSS】 ====================

liquid_glass_css = """
body, .gradio-container {
    background-color: #1A1919 !important;
    background-image: radial-gradient(circle at 50% 0%, #242120 0%, #1A1919 80%) !important;
    color: #E2E8F0 !important;
    font-family: system-ui, -apple-system, sans-serif !important;
}
.gr-box, .gr-panel, .gr-form { border: none !important; background: transparent !important; box-shadow: none !important; }

/* 強制隱藏所有原生工具列與右上角叉叉 */
button[aria-label="Clear"], button[aria-label="Copy"], .gr-button-icon, .close-button {
    display: none !important;
}

.glass-panel {
    background: rgba(255, 255, 255, 0.015) !important;
    backdrop-filter: blur(16px) !important;
    border: 1px solid rgba(255, 255, 255, 0.03) !important;
    border-top: 1px solid rgba(255, 255, 255, 0.07) !important;
    border-radius: 16px !important;
    padding: 20px !important;
    box-shadow: 0 12px 40px rgba(0, 0, 0, 0.4) !important;
}

.glass-card {
    background: linear-gradient(135deg, rgba(255, 255, 255, 0.025) 0%, rgba(255, 255, 255, 0.005) 100%) !important;
    backdrop-filter: blur(12px) !important;
    border: 1px solid rgba(255, 255, 255, 0.03) !important;
    border-top: 1px solid rgba(212, 175, 55, 0.18) !important;
    border-radius: 10px !important;
    padding: 14px !important;
}

.ai-box {
    background: rgba(212, 175, 55, 0.015) !important;
    border: 1px solid rgba(212, 175, 55, 0.1) !important;
    border-radius: 10px !important;
    font-size: 14px !important;
    color: #F6F0E5 !important;
    padding: 12px !important;
}

.accent-btn {
    background: linear-gradient(135deg, #D4AF37 0%, #AA6C39 100%) !important;
    color: #1A1919 !important;
    border: none !important;
    border-radius: 6px !important;
    font-weight: 600 !important;
}
.accent-btn:hover {
    background: linear-gradient(135deg, #E5C158 0%, #BD7D4A 100%) !important;
    transform: translateY(-1px) !important;
}

input[type="range"]::-webkit-slider-thumb { background: #D4AF37 !important; }
"""

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="amber", neutral_hue="stone"),
    css=liquid_glass_css
) as demo:

    gr.Markdown("## 訓練課表產生器", elem_classes="gr-box")

    with gr.Row():
        # 【左側：參數面板】
        with gr.Column(scale=1, elem_classes="glass-panel"):
            gr.Markdown("#### 參數設定")
            inp_sport = gr.Radio(["跑步 (Running)", "騎車 (Cycling)", "游泳 (Swimming)"], label="運動項目", value="跑步 (Running)")
            inp_run_type = gr.Dropdown(["有氧基礎跑 (E 跑)", "馬拉松配速跑 (M 跑)", "乳酸閥值/節奏跑 (T 跑)", "最大攝氧量間歇跑 (I 跑)"], label="跑步訓練類型", value="有氧基礎跑 (E 跑)", visible=True)

            # 🛠️ 【全新功能】：讓使用者輸入個人化配速，作為整套系統的配速計算基準底蘊
            inp_personal_pace = gr.Textbox(label="個人馬拉松目標配速 (分:秒)", value="05:30", placeholder="例如 05:30", visible=True)

            inp_fatigue = gr.Slider(minimum=1, maximum=10, step=1, label="今日疲勞度指標 (1-10)", value=3)
            inp_time = gr.Slider(minimum=20, maximum=180, step=5, label="預計總時間 (分鐘)", value=60)
            btn_submit = gr.Button("生成訓練計畫", elem_classes="accent-btn")

        # 連動控制：切換運動項目時，動態決定是否顯示個人跑步配速輸入框
        def update_visibility_wrapper(sport):
            is_run = "跑步" in sport
            return gr.update(visible=is_run), gr.update(visible=is_run)

        inp_sport.change(fn=update_visibility_wrapper, inputs=inp_sport, outputs=[inp_run_type, inp_personal_pace])

        # 【右側：結果顯示面板】
        with gr.Column(scale=2, elem_classes="glass-panel"):
            gr.Markdown("#### 系統體能診斷")
            out_ai = gr.Textbox(label="", lines=2, elem_classes="ai-box", interactive=False, show_copy_button=False)

            gr.Markdown("#### 結構化課表")
            with gr.Row():
                with gr.Column(scale=1, min_width=0):
                    out_warmup = gr.Textbox(label="1. 暖身階段", lines=6, elem_classes="glass-card", interactive=False, show_copy_button=False)
                with gr.Column(scale=1, min_width=0):
                    out_main = gr.Textbox(label="2. 主訓練階段", lines=6, elem_classes="glass-card", interactive=False, show_copy_button=False)
                with gr.Column(scale=1, min_width=0):
                    out_cooldown = gr.Textbox(label="3. 緩和階段", lines=6, elem_classes="glass-card", interactive=False, show_copy_button=False)

            gr.Markdown("#### 強度分佈趨勢")
            out_chart = gr.Plot(label="", show_label=False)

    # 1. 點擊按鈕事件（加入 inp_personal_pace 傳入後端）
    btn_submit.click(
        fn=calculate_pace_and_structure,
        inputs=[inp_sport, inp_run_type, inp_fatigue, inp_time, inp_personal_pace],
        outputs=[out_ai, out_warmup, out_main, out_cooldown, out_chart]
    )

    # 2. 頁面載入自動觸發
    demo.load(
        fn=calculate_pace_and_structure,
        inputs=[inp_sport, inp_run_type, fatigue := inp_fatigue, inp_time, inp_personal_pace],
        outputs=[out_ai, out_warmup, out_main, out_cooldown, out_chart]
    )

demo.launch(share=True)

/tmp/ipykernel_7623/1313512298.py:269: DeprecationWarning:

The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.

/tmp/ipykernel_7623/1313512298.py:269: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://62c7dcb0f8f2a10d2b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
